# Structure-Based HMM Profiling of Kunitz-Type Domains

This notebook outlines a complete bioinformatics workflow for detecting Kunitz-type serine protease inhibitor domains (PF00014) in protein sequences using Hidden Markov Models (HMMs).

The pipeline includes the following key steps:
- Positive and negative protein sequences are gathered and prepared for analysis.
- CD-HIT is employed to remove highly similar sequences, ensuring a non-redundant dataset.
- Experimentally resolved structures from the PDB are used to generate a structure-guided multiple sequence alignment.
- A profile HMM is generated from the refined structural alignment using hmmbuild.
- Sequences used for training are excluded from testing to prevent evaluation bias.
- The trained model is applied to test datasets using hmmsearch to identify potential Kunitz domains.
- The model is evaluated across various e-value thresholds to assess classification accuracy and robustness.



## 1. Separation of Human and Non-Human Kunitz sequences
To begin, sequences are classified based on species annotation in the `all_kunitz.fasta` file. Specifically, entries containing "Homo sapiens" are extracted to create a human-specific dataset, while all other entries are saved as non-human sequences.

In [ ]:
# Human Kunitz sequences
awk '/>/{f=($0 ~ /Homo sapiens/)} f' all_kunitz.fasta > human_kunitz_sequences.fasta

# Non-Human Kunitz sequences
grep -v "Homo sapiens" all_kunitz.fasta > non_human_kunitz_sequences.fasta

## 2. Processing PDB sequences annotated with PF00014
To build a curated structural dataset, a custom CSV report was first generated from the RCSB PDB by selecting entries annotated with the PF00014 domain. The report was further refined by applying filters on sequence length (45–80 amino acids) and structural resolution (≤ 3.5 Å).

Once downloaded, the report was parsed to extract relevant sequence fields and convert them into FASTA format. These sequences were then subjected to redundancy filtering using CD-HIT, with a 90% identity threshold to retain only representative entries. Finally, known outliers were removed manually to ensure high-quality input for downstream analysis

In [ ]:
# Clean and convert the CSV report into FASTA format (only PF00014 entries)
cat rcsb_pdb_custom_report_20250505025420.csv | tr -d '"' \
  | awk -F ',' '{if (length($2)>0) {name=$2}; print name ,$3,$4,$5}' \
  | grep PF00014 \
  | awk '{print ">"$1"\n"$3; print $2}' > pdb_kunitz_customreported.fasta

# Cluster sequences at 90% identity to reduce redundancy
cd-hit -i pdb_kunitz_customreported.fasta -o pdb_kunitz_customreported_nr.fasta -c 0.9

# Remove specific low-quality sequences (2ODY_E) from the clustered dataset
awk '/^>2ODY_E/ {getline; next} {print}' pdb_kunitz_customreported_nr.fasta > pdb_kunitz_customreported_filtered.fasta

# Apply the same filtering logic to the CD-HIT clustering report (.clstr)
awk 'BEGIN{skip=0}/^>cluster 0$/ {skip=1; next} /^>cluster/ {skip=0} !skip' pdb_kunitz_customreported_nr.fasta.clstr > pdb_kunitz_customreported_filtered.clstr


## 3. Identify and retrieve cluster representatives
After clustering with CD-HIT, one representative sequence per cluster is selected for structural analysis. This step involves converting the `.clstr` file into a human-readable format, extracting representative sequence IDs, retrieving the corresponding sequences from the original FASTA file, and reformatting the identifiers for compatibility with PDBeFold.

In [ ]:
# Convert the CD-HIT .clstr file into a tabular format for easier parsing
clstr2txt.pl pdb_kunitz_customreported_filtered.clstr > pdb_kunitz.clusters.txt

# Extract representative sequence IDs (identified by '1' in the fifth column)
awk '$5 == 1 {print $1}' pdb_kunitz.clusters.txt > pdb_kunitz_rp.ids

# Retrieve the corresponding FASTA sequences for the selected representative IDs
> pdb_kunitz_rp.fasta
for i in $(cat pdb_kunitz_rp.ids); do
  grep -A 1 ">$i" pdb_kunitz_customreported.fasta | head -n 2 >> pdb_kunitz_rp.fasta
done

# Convert sequence IDs to the format required by PDBeFold (underscores replaced by colons)
grep "^>" pdb_kunitz_rp.fasta | tr -d ">" | tr "_" ":" > tmp_pdb_efold_ids.txt


## 4. Formatting of structural alignment and HMM generation
The structural alignment obtained from PDBeFold in `.ali` format requires reformatting to meet the input requirements of the `hmmbuild` tool from the HMMER suite. This involves converting the alignment into uppercase FASTA-style format, ensuring compatibility for model construction.

In [ ]:
# Reformat the .ali file: convert to uppercase and insert FASTA-style line breaks
awk '{
  if (substr($1,1,1)==">") {
    print "\n" toupper($1)
  } else {
    printf "%s", toupper($1)
  }
}' pdb_kunitz_rp.ali > pdb_kunitz_rp_formatted.ali


After formatting, the profile Hidden Markov Model is built from the structural alignment:

In [ ]:
hmmbuild structural_model.hmm pdb_kunitz_rp_formatted.ali

## 5. Removal of redundant matches via BLAST filtering
To ensure an unbiased evaluation of the model, sequences that are too similar to those used for HMM construction must be excluded from the test set. A BLAST comparison is performed between the representative PDB sequences and the full Kunitz dataset from UniProt. Entries with ≥95% identity and ≥50% coverage are discarded.

In [ ]:
# Format the full Kunitz dataset as a BLAST database
makeblastdb -in all_kunitz.fasta -dbtype prot -out all_kunitz.fasta

# Compare representative sequences (query) against the full dataset (database)
blastp -query pdb_kunitz_rp.fasta -db all_kunitz.fasta -out pdb_kunitz_nr_23.blast -outfmt 7


From the BLAST output, overlapping hits are identified and removed:

In [ ]:
# Identify UniProt IDs with high similarity (≥95% identity and ≥50% coverage)
grep -v "^#" pdb_kunitz_nr_23.blast | awk '{if ($3>=95 && $4>=50) print $2}' \
  | sort -u | cut -d "|" -f 2 > to_remove.ids

# Retrieve all UniProt IDs from the complete dataset for filtering
grep "^>" all_kunitz.fasta | cut -d "|" -f 2 > all_kunitz.id

Remaining sequences are selected by excluding those marked for removal:

In [ ]:
# Generate a list of non-overlapping UniProt IDs (retain only novel entries)
comm -23 <(sort all_kunitz.id) <(sort to_remove.ids) > to_keep.ids

# Extract the final positive sequences for model testing
python3 get_seq.py to_keep.ids all_kunitz.fasta ok_kunitz.fasta


## 6. Generation of the negative sequence dataset
To assemble a negative dataset, protein sequences are filtered to exclude any entries associated with the Kunitz domain. The resulting set consists of proteins unlikely to contain the target domain, providing a background for evaluating model specificity.

In [ ]:
# Extract all UniProt IDs from the Swiss-Prot dataset
grep "^>" uniprot_sprot.fasta | cut -d "|" -f 2 > sp.id

# Identify Swiss-Prot entries that are not part of the Kunitz dataset
comm -23 <(sort sp.id) <(sort all_kunitz.id) > sp_negs.ids

# Retrieve sequences corresponding to the filtered list of IDs
python3 get_seq.py sp_negs.ids uniprot_sprot.fasta sp_negs.fasta


## 7. Construction of Training and Testing Sets
To enable model validation, both the positive and negative datasets are randomly divided into two equal subsets. This procedure ensures that one subset can be used for model training while the other serves as an independent test set. The split is performed at the level of sequence identifiers, followed by extraction of the corresponding FASTA entries.

In [ ]:
# Randomize the order of sequence IDs in both datasets
sort -R to_keep.ids > random_ok_kunitz.ids
sort -R sp_negs.ids > random_sp_negs.ids

# Split the randomized ID lists into two equal subset (for cross-validation)
head -n 183 random_ok_kunitz.ids > pos_1.ids
tail -n 183 random_ok_kunitz.ids > pos_2.ids

head -n 286417 random_sp_negs.ids > neg_1.ids
tail -n 286417 random_sp_negs.ids > neg_2.ids

# Extract FASTA sequences corresponding to each subset
python3 get_seq.py pos_1.ids uniprot_sprot.fasta > pos_1.fasta
python3 get_seq.py pos_2.ids uniprot_sprot.fasta > pos_2.fasta
python3 get_seq.py neg_1.ids uniprot_sprot.fasta > neg_1.fasta
python3 get_seq.py neg_2.ids uniprot_sprot.fasta > neg_2.fasta

## 8. Domain search with HMMER and preparation of classification files
Once the structural profile HMM has been constructed, it is used to scan the positive and negative datasets to assess how well the model discriminates between true Kunitz sequences and unrelated proteins.

To do so, the `hmmsearch` tool from the HMMER suite is executed on each of the four test sets (`pos_1.fasta`, `pos_2.fasta`, `neg_1.fasta`,`neg_2.fasta`). The `--tblout` option produces a tabular summary of the hits, while the `-Z 1000` flag ensures that e-values are scaled consistently across datasets of varying sizes by setting the effective database size to 1000. This normalization is important for making e-values directly comparable between runs.

In [ ]:
# # Perform domain search using the structural HMM
hmmsearch -Z 1000 --max --tblout pos_1.out structural_model.hmm pos_1.fasta
hmmsearch -Z 1000 --max --tblout pos_2.out structural_model.hmm pos_2.fasta
hmmsearch -Z 1000 --max --tblout neg_1.out structural_model.hmm neg_1.fasta
hmmsearch -Z 1000 --max --tblout neg_2.out structural_model.hmm neg_2.fasta

After the search, the relevant information is parsed from each output file to generate .class files, which serve as standardized inputs for performance evaluation scripts (e.g., performance.py).
Each line in a .class file corresponds to one sequence and contains the following fields:
1. UniProt ID
2. Label (1 for positive sequences, 0 for negative)
3. Full-sequence e-value (reported by HMMER)
4. Domain-level e-value (if available, otherwise .)

In [ ]:
# Parse output and convert to .class format for POSITIVE sets
grep -v "^#" pos_1.out | awk '{split($1,a,"|"); print a[2]"\t1\t"$5"\t"$8}' > pos_1.class
grep -v "^#" pos_2.out | awk '{split($1,a,"|"); print a[2]"\t1\t"$5"\t"$8}' > pos_2.class

#  Parse output and convert to .class format for NEGATIVE sets
grep -v "^#" neg_1.out | awk '{split($1,a,"|"); print a[2]"\t0\t"$5"\t"$8}' > neg_1.class
grep -v "^#" neg_2.out | awk '{split($1,a,"|"); print a[2]"\t0\t"$5"\t"$8}' > neg_2.class


However, not all sequences will produce a hit in the `hmmsearch` results. To ensure that these sequences are still evaluated (especially important for negatives), they are added manually to the `.class` files with a placeholder e-value of 10.0. This conservative value reflects a very weak or absent match.

In [ ]:
comm -23 <(sort neg_1.ids) <(cut -f1 neg_1.class | sort) | awk '{print $1"\t0\t10.0\t10.0"}' >> neg_1.class
comm -23 <(sort neg_2.ids) <(cut -f1 neg_2.class | sort) | awk '{print $1"\t0\t10.0\t10.0"}' >> neg_2.class

This results in complete `.class` files for each dataset, ensuring that every sequence is represented, whether or not it produced a significant match. 

## 9. Dataset merging for evaluation
To prepare the data for downstream classification performance evaluation, the previously generated `.class` files for positive and negative samples are combined into two unified datasets. These merged files represent the full content of each test fold and are referred to as `set_1.class` and `set_2.class`.

Each `.class` file contains tab-separated values including the sequence ID, binary label (1 for positive, 0 for negative), full-sequence e-value, and domain e-value.

In [ ]:
# Combine positive and negative class files for Fold 1
cat pos_1.class neg_1.class > set_1.class

# Combine positive and negative class files for Fold 2
cat pos_2.class neg_2.class > set_2.class

These final `.class` files can now be used as input to performance analysis scripts to calculate different classification metrics.

## 10. Model evaluation across E-Value thresholds
To assess model performance, the `.class` files (`set_1.class` and `set_2.class`) are evaluated using the `performance.py` script, which computes classification metrics such as precision, recall, and the Matthews Correlation Coefficient (MCC) at a given e-value threshold.

Initially, evaluation is performed at a fixed threshold (1e-5), but to explore the model’s sensitivity and determine the optimal decision boundary, the analysis is repeated across a range of thresholds from 1e-1 to 1e-10.

In [ ]:
# Evaluate classification performance at threshold 1e-5
python3 performance.py set_1.class 1e-5
python3 performance.py set_2.class 1e-5
  
# Iterate over thresholds for set_1 (from 1e-1 to 1e-10)
for i in $(seq 1 10); do
  python3 performance.py set_1.class 1e-$i
done > performance_set1_thresholds.txt

# Iterate over thresholds for set_2 (from 1e-1 to 1e-10)
for i in $(seq 1 10); do
  python3 performance.py set_2.class 1e-$i
done > performance_set2_thresholds.txt

This approach allows identifying the threshold that maximizes MCC and provides the best balance between sensitivity and specificity.

## 11. Error Analysis – Identification of False Negatives and False Positives
To better understand the model’s limitations, it is useful to inspect misclassified sequences—particularly false negatives (positives missed by the model) and false positives (negatives incorrectly classified as positives).

The `.class` files are filtered based on e-value thresholds. In this context, sequences with e-values greater than the selected cutoff (1e-5) are considered missed, while those below the threshold are considered detected.

In [ ]:
# Identify false negatives: true positives (label 1) with poor scores (e-value > 1e-5)
awk '$2 == 1 && $3 > 1e-5' pos_1.class | sort -grk 3 > fn_pos1.txt
awk '$2 == 1 && $3 > 1e-5' pos_2.class | sort -grk 3 > fn_pos2.txt

# Identify false positives: true negatives (label 0) with unexpectedly good scores (e-value < 1e-5)
awk '$2 == 0 && $3 < 1e-5' neg_1.class | sort -grk 3 > fp_neg1.txt
awk '$2 == 0 && $3 < 1e-5' neg_2.class | sort -grk 3 > fp_neg2.txt

These files can be used for manual review, functional annotation checks, or structural inspection of ambiguous cases. This kind of error analysis is critical for understanding the failure modes of the model and guiding future improvements.